In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from typing import cast
import warnings

import arviz_plots as azp
import arviz_stats as azs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from compressor_fouling_modeling.utility import (
    CoefficientStructure,
    CoefPrior,
    InterceptStructure,
    LikeLiHood,
    NoiseStructure,
    build_and_fit_bayesian_models,
    calculate_cusum_with_uncertainty,
    calculate_empirical_sigma_stats,
    check_likelihood_qqplot,
    compare_and_select_best_model,
    compute_exceedance_probability,
    compute_masked_statistics,
    compute_psis_weights,
    evaluate_model_elpd,
    evaluate_model_performance,
    evaluate_noise_model,
    exctract_pymc_groups_data,
    plot_loo_calibration_curves,
    plot_posterior_predictive,
    plot_predictions_with_uncertainty,
    prepare_bayesian_model_args,
    prepare_hierarchical_noise_args,
    visualize_density_clusters,
    visualize_probabilistic_cusum,
)

os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # makes JAX allocate exactly what is needed on demand
# azb.rcParams to check default parameters in arviz base
# azp.style.available() to check available stylesof interest
azp.style.use("dark_background")  # pick style of interest
%config InlineBackend.figure_format = 'retina'  # high resolution figures
warnings.filterwarnings("ignore")

In [ ]:
RANDOM_SEED = 14
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
# Define project root relative to notebook location
PROJECT_ROOT = Path().resolve().parents[0]  # goes up one level from /notebooks/
IMAGE_DIR = Path(PROJECT_ROOT / "results" / "plots")
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
# Check if the directory exists and create it if it doesn't
if not IMAGE_DIR.exists():
    try:
        IMAGE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Directory '{IMAGE_DIR}' created successfully.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"Directory '{IMAGE_DIR}' already exists.")

In [ ]:
X_baseline = pd.read_csv(DATA_DIR / "processed" / "X_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")
y_baseline = pd.read_csv(DATA_DIR / "processed" / "y_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")
X = pd.read_csv(DATA_DIR / "processed" / "X_full.csv", index_col=0, parse_dates=True).squeeze("columns")
y = pd.read_csv(DATA_DIR / "processed" / "y_full.csv", index_col=0, parse_dates=True).squeeze("columns")
baseline_mask = pd.read_csv(DATA_DIR / "processed" / "baseline_mask.csv", index_col=0, parse_dates=True).squeeze("columns")
shutin_mask = pd.read_csv(DATA_DIR / "processed" / "shutin_mask.csv", index_col=0, parse_dates=True).squeeze("columns")

# Bayesian Approach

In [ ]:
setpoint_unique, setpoint_index, map_sp_to_idx = prepare_hierarchical_noise_args(
    X_baseline
)
empirical_stats, mean_std, range_to_mean_ratio, min_std, max_std = (
    calculate_empirical_sigma_stats(X_baseline, y_baseline, setpoint_unique.tolist())
)
std_unscaled = {sp: empirical_stats[sp]["std"] for sp in empirical_stats}

In [ ]:
print(
    "Prepare the dataset for regressing outlet-pressure residuals against their"
    " respective setpoints."
)
data_residual = prepare_bayesian_model_args(
    X_baseline, y_baseline, shuffle_baseline=False, residual_target=True
)
lr = LinearRegression()
lr.fit(data_residual.X_scaled, data_residual.y_scaled)
y_pred = lr.predict(data_residual.X_scaled)
r2 = r2_score(data_residual.y_scaled, y_pred)
print(
    f"\nRoughly {r2:.3f} of variance can be explained by features altogether after"
    " removing outlet_pressure_sp."
)

fig, ax = plt.subplots(1, 1)
ax.hist(data_residual.y_scaled, density=True)
ax.set_title("target (outlet_pressure - outlet_pressure_sp) distribution")
plt.show()
plt.close(fig)
del fig, ax

The problem that we are trying to solve is:

$$\textbf{\textit{outlet\_pressue}} = \textbf{\textit{noise}} + \textbf{\textit{intercept}} + \textbf{\textit{outlet\_pressure\_SP}} \times \beta_{outlet\_pressure\_sp} + \textbf{\textit{Inlet\_Temperature}} \times \beta_{Inlet\_Temperature} + \dots$$

The correct approach is to regress $\textit{outlet\_pressue}$ on all predictors as stated above and find the respective $\beta$ of each predictor. However we know in a controlled system like separator, the setpoints are dominant variables. For example, the $\textit{outlet\_pressue}$ for the most part should follow $\textit{outlet\_pressure\_sp}$. Thus one may mistakenly consider $\beta_{outlet\_pressure\_sp}=1$ and regress on other variables in two ways:

1. Consider $\textbf{\textit{outlet\_pressure\_SP}}$ as an offset on the righ-hand-side of equation and consider all other predictors adjust this offset to match $\textbf{\textit{outlet\_pressue}}$ changes.
2. Move $\textbf{\textit{outlet\_pressure\_SP}}$ to the left-hand-side and in fact model the direct resdiuals $\textbf{\textit{outlet\_pressue}} - \textbf{\textit{outlet\_pressure\_SP}}$ using remaining predictors.

We do not follow these two approaches because:
1. Considering $\beta_{outlet\_pressure\_sp}=1$ introduces bias if the value is not in fact very close to 1 which is most likley the case.
2. Although we expect the $\beta_{outlet\_pressure\_sp}$ not to be 1, it should be very high as it is the nature of such controlled processes. Thus, removing it from predictors reduces variance in data significantly which results in model under performance.

In [ ]:
print("\nPrepare the dataset for regressing outlet-pressure directly.")
data_out_pressure = prepare_bayesian_model_args(
    X_baseline, y_baseline, shuffle_baseline=True, random_seed=RANDOM_SEED
)

In [ ]:
visualize_density_clusters(
    data_out_pressure.y_scaled.to_numpy(),
    bound_width=mean_std / data_out_pressure.y_std,
    hist_bins=50,
)

In [ ]:
noise_kwargs = {
    "sigma_mu_mu": mean_std / data_out_pressure.y_std,
    "sigma_mu_sd": np.log(1 + 1.0 * range_to_mean_ratio),  # ~12% variation around
    # mean according to np.exp(sigma_mu_sd)
}

hierarchical_noise_kwargs = {
    "setpoint_unique": setpoint_unique,
    "setpoint_index": setpoint_index,
    "sigma_sd_sd": np.log(
        1 + 1.5 * range_to_mean_ratio
    ),  # ~18% between-setpoint variation (24% total multiplicative uncertanity
    # according to exp(sqrt(0.12**2+0.18**2)))
}

coef_kwargs = {"mu": 0.0, "sd": 0.25}
likelihood_model_kwargs = {"alpha": 5.0, "beta": 0.25}  # for T-student likelihood

# Define model configurations
model_configs = [
    {
        "name": "non_hierarchical",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
    },
    {
        "name": "all_hierachical",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "has_setpoint_coords": True,
        "noise_structure": NoiseStructure.HIERARCHICAL,
        "intercept_structure": InterceptStructure.HIERARCHICAL,
        "coefficient_structure": CoefficientStructure.HIERARCHICAL,
    },
    {
        "name": "all_hierachical_but_coefficients",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "has_setpoint_coords": True,
        "noise_structure": NoiseStructure.HIERARCHICAL,
        "intercept_structure": InterceptStructure.HIERARCHICAL,
    },
    {
        "name": "hierarchical_noise",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "has_setpoint_coords": True,
        "noise_structure": NoiseStructure.HIERARCHICAL,
    },
    {
        "name": "hierarchical_intercept",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "has_setpoint_coords": True,
        "intercept_structure": InterceptStructure.HIERARCHICAL,
    },
    {
        "name": "non_hierarchical_laplace_prior",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "coef_prior": CoefPrior.LAPLACE,
    },
    {
        "name": "all_hierachical_but_coefficients_laplace_prior",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "has_setpoint_coords": True,
        "noise_structure": NoiseStructure.HIERARCHICAL,
        "intercept_structure": InterceptStructure.HIERARCHICAL,
        "coef_prior": CoefPrior.LAPLACE,
    },
    {
        "name": "non_hierarchical_laplace_prior_t_Likelihood",
        "X": data_out_pressure.X_scaled,
        "y": data_out_pressure.y_scaled,
        "coef_prior": CoefPrior.LAPLACE,
        "likelihood_model": LikeLiHood.T,
    },
]

# Common parameters for all models
common_params = {
    "intercept_sd": 0.4,
    "coef_kwargs": coef_kwargs,
    "noise_kwargs": noise_kwargs,
    "hierarchical_kwargs": hierarchical_noise_kwargs,
    "likelihood_model_kwargs": likelihood_model_kwargs,
    "random_seed": RANDOM_SEED,
}

model_dict, idatas_dict = build_and_fit_bayesian_models(model_configs, common_params)

In [ ]:
model_top, idata_top = compare_and_select_best_model(model_dict, idatas_dict)

The Laplace prior is winning because it's doing feature selection - it's shrinking irrelevant coefficients harder than Normal prior, which is why it generalizes better. Regarding the noise model, single noise model outperforms the hierarchical noise model likely as the difference between sigms for multiple setpoints is not significant enough (minimum and maximum sigmas differ from averge sigma just 7% and 10%, respectively!) to be propely modeled particularly given the limited number of samples. 

In [ ]:
# prior knowledge
y_min = 5  # minimum plausible compressor pressure (psi)
y_max = 250  # maximum plausible compressor pressure (psi)
# check if prior is not too far from the data distribution
pc = azp.plot_ppc_dist(
    idata_top,
    group="prior_predictive",
    kind="ecdf",
    visuals={"predictive_dist": {"color": "C1"}, "observed_dist": {"color": "C3"}},
    num_samples=1000,
)
azp.add_lines(
    pc,
    values=(
        (y_min - data_out_pressure.y_mean) / data_out_pressure.y_std,
        (y_max - data_out_pressure.y_mean) / data_out_pressure.y_std,
    ),
)
pc.show()

In [ ]:
node_formatters = {
    "Free Random Variable": lambda var: {"shape": "circle", "label": var.name},
    "Observed Random Variable": lambda var: {"shape": "square", "label": var.name},
}
pm.model_to_graphviz(
    model_top,
    node_formatters=node_formatters,  # type: ignore
)

In [ ]:
pc = azp.plot_energy(idata_top, kind="ecdf")

# Warning not to use ArviZ KDE in multimodal distribution

What ArviZ KDE Does

Step 1: For posterior draw s, take 130 predictive samples
        ỹ_i ~ Normal(μ_i^(s), σ^(s))

Step 2: Apply KDE to these 130 points
        → KDE auto-selects bandwidth based on overall data spread

The problem is in Step 2. Standard bandwidth selectors (Scott/Silverman) assume unimodal data:

Silverman's rule
bw = 0.9 * min(std, IQR/1.34) * n**(-1/5)
data: std ≈ 1.0, n = 130
bw ≈ 0.9 * 1.0 * 130**(-0.2) ≈ 0.34

So the KDE uses bandwidth ≈ 0.34, while the actual within-cluster spread is σ = 0.074:

KDE bandwidth:    0.34  ← smears clusters together
Actual σ:         0.074 ← sharp, separated clusters

KDE is ~5× wider than reality!

In [ ]:
# ecdf is more informative here than the default kde as it is not bandwidth
# sensitive and the data distribution is not unimodal normal
# band_width = mean_std / data_out_pressure.y_std
# pc = azp.plot_ppc_dist(
#     idata_top,
#     kind="kde",
#     stats={"predictive_dist": {"bw": band_width}, "observed_dist": {"bw": band_width}},
#     num_samples=1000,
#     figure_kwargs={"figsize": (10, 5)},
# )

In [ ]:
fig_ppc_ecdf_kde, fig_ppc_obs_kde = plot_posterior_predictive(
    idata_top, random_seed=RANDOM_SEED
)

In [ ]:
pc = azp.plot_prior_posterior(
    idata_top,
    var_names=["intercept", "beta", "sigma", "tau"],
    kind="ecdf",
)
pc = azp.plot_ppc_interval(
    idata_top,
    ci_kind="hdi",
    ci_probs=(0.925, 0.975),
    figure_kwargs={"figsize": (10, 5)},
    backend="plotly",
)
pc.show()

In [ ]:
pc = azp.combine_plots(
    idata_top,
    plots=[
        (azp.plot_ppc_tstat, {"t_stat": "median"}),
        (azp.plot_ppc_tstat, {"t_stat": "mad"}),
        (azp.plot_ppc_tstat, {"t_stat": "iqr"}),
    ],
    group="posterior_predictive",
)

In [ ]:
pc = azp.plot_trace_dist(
    idata_top,
    var_names=["intercept", "beta", "sigma"],
    compact=True,
    combined=False,
)

In [ ]:
qqplot_normal_likelihhod = check_likelihood_qqplot(
    idata_top, "normal", data_out_pressure.y_scaled.to_numpy()
)

The Q-Q plot shows:

✅ Very good fit to normality - points fall very close to the 45° line
✅ R² = 0.9660 - extremely high linearity
✅ No systematic deviations - no S-curves, no heavy tails, no obvious patterns
✅ Only minor deviations at extremes - a couple of points slightly off at the very ends (around -2 and +2), which is typical and acceptable

What the slight tail deviations mean:

The few points deviating at the extremes (one at ~-2.1 and a couple at ~+1.8 to +2.0) suggest very slight heavy tails, but this is extremely minor. In practice, this level of deviation is:

Well within acceptable bounds for real-world data

Not significant enough to warrant switching to Student's t
Could just be natural sampling variability with only 130 observations

Conclusion:

The Normal likelihood is well-justified. The residuals are approximately normally distributed, which validates the modeling choice. You don't need to change to a more complex likelihood (Student's t, mixture, etc.).

In [ ]:
idata_top_data = exctract_pymc_groups_data(idata_top)

In [ ]:
rope = [(-0.05, 0.05)]
pc = azp.plot_forest(
    idata_top,
    var_names=["tau", "intercept", "beta", "sigma"],
    combined=True,
    ci_kind="hdi",
    ci_probs=(0.925, 0.975),
)
pc.coords = {"column": "forest"}
pc = azp.add_bands(
    pc,
    values=rope,
    visuals={"ref_band": {"color": "C1"}},
)

In [ ]:
# pc = azp.plot_loo_pit(idata_top, coverage=True, figure_kwargs={"figsize": (6, 3)})
out_pressure_loo = azs.loo(idata_top, pointwise=True)
out_pressure_loo

In [ ]:
with model_top:
    print(f"list of free (unconstrained) variables: {model_top.free_RVs}")
    # For total scalar count:
    n_params = sum(rv.size.eval() for rv in model_top.free_RVs)
    print(f"Total scalar parameters: {n_params}")

In [ ]:
# elpd_in_sample = Σ_i log[E_post(p(y_i|θ))]
elpd_in_sample = np.log(
    np.exp(idata_top.log_likelihood["y_like"]).stack(s=("chain", "draw")).mean(dim="s")  # noqa: PD013 - stack() required for xarray dimension stacking
).sum()

# from scipy.stats import norm

# idata = trace_out_pressure
# ## Extract posterior samples
# mu_samples = idata.posterior["mu"].values.reshape(-1, 130)  # (S, N)
# sigma_samples = np.exp(idata.posterior["log_sigma"].values.reshape(-1, 1))  # (S, 1)
# S = mu_samples.shape[0]  # 8000 samples
# # For each observation, compute average likelihood over posterior samples
# # then take log — this is the exact MCMC approximation of ELPD
# log_avg_likelihood = np.zeros(130)
# for i in range(130):
#     # p(y_i | theta^j) for all j
#     likelihoods = norm.pdf(
#         y_obs.to_numpy()[i], loc=mu_samples[:, i], scale=sigma_samples[:, 0]
#     ) # (chain * draw, N)
#     # average over S samples (chain * draw), then log
#     log_avg_likelihood[i] = np.log(np.mean(likelihoods))

# elpd_in_sample = np.sum(log_avg_likelihood)

optimism = elpd_in_sample - out_pressure_loo.elpd
optimism_ratio = optimism / len(data_out_pressure.y_scaled)

print(f"in-sample ELPD:                            {elpd_in_sample:.2f}")
print(f"LOO ELPD (out-of-sample):                  {out_pressure_loo.elpd:.2f}")
print(f"Total optimism (overfitting tendency) or p-loo:         {optimism:.2f}")
print(
    "individual contributions of each observation to the overfitting tendency: "
    f"{optimism_ratio * 100:.2f}%"
)

# Understanding the Expected Log Pointwise Predictive Density (elpd_loo)

## 1. Definition

The **Expected Log Pointwise Predictive Density** using Leave-One-Out cross-validation (elpd_loo) is defined as:

$$\text{elpd\_loo} = \sum_{i=1}^{n} \log p(y_i \mid y_{-i})$$

where:
- $y_i$ is observation $i$
- $y_{-i}$ represents all observations *except* observation $i$
- $p(y_i \mid y_{-i})$ is the posterior predictive density for $y_i$ when the model is trained on all data except $y_i$

This metric quantifies how well the model predicts held-out observations, providing an estimate of out-of-sample predictive performance.

---

## 2. Results Interpretation

For our Bayesian regression model, the LOO cross-validation results are:

| Metric | Estimate | SE |
|--------|----------|-----|
| elpd_loo | 148.60 | 5.35 |
| p_loo | 9.71 | -- |

*Computed from 16,000 posterior samples and 86 observations.*

### Per-Observation Average

The average log predictive density per observation is:

$$\overline{\text{elpd}} = \frac{\text{elpd\_loo}}{n} = \frac{148.60}{130} \approx 1.15$$

---

## 3. Mathematical Derivation of Expected Values

For a Normal likelihood, the log predictive density for observation $y_i$ given predicted mean $\mu_i$ and standard deviation $\sigma$ is:

$$\log p(y_i \mid \mu_i, \sigma) = -\frac{1}{2}\log(2\pi) - \log(\sigma) - \frac{(y_i - \mu_i)^2}{2\sigma^2}$$

### Baseline 1: Null Model (No Predictive Power)

For standardized data ($y \sim \mathcal{N}(0, 1)$), a null model predicts $\mu = 0$ and $\sigma = 1$ for all observations:

$$\log p(y_i \mid \mu=0, \sigma=1) = -\frac{1}{2}\log(2\pi) - \log(1) - \frac{y_i^2}{2}$$

Taking the expectation when $y_i \sim \mathcal{N}(0, 1)$, we have $\mathbb{E}[y_i^2] = 1$:

$$\mathbb{E}[\log p(y)] = -\frac{1}{2}\log(2\pi) - 0 - \frac{1}{2} = -0.919 - 0.5 = \mathbf{-1.42}$$
<!--  -->
### Baseline 2: Our Fitted Model

Our model achieves a residual standard deviation of $\sigma \approx 0.07$, meaning predictions are very close to true values. For a well-calibrated model where residuals follow $(y_i - \hat{y}_i) \sim \mathcal{N}(0, \sigma^2)$:

$$\log p(y_i \mid \hat{y}_i, \sigma=0.07) = -\frac{1}{2}\log(2\pi) - \log(0.07) - \frac{(y_i - \hat{y}_i)^2}{2(0.07)^2}$$

Taking the expectation:

$$\mathbb{E}[\log p] = -\frac{1}{2}\log(2\pi) - \log(0.07) - \frac{\sigma^2}{2\sigma^2}$$

$$= -0.919 + 2.66 - 0.5 = \mathbf{+1.18}$$

This theoretical value closely matches our observed average of $+1.15$.

### Baseline 3: Perfect Model

In the limit of perfect predictions ($\sigma \to 0$):

$$\lim_{\sigma \to 0} \log p(y_i \mid \hat{y}_i, \sigma) = +\infty$$

This is unattainable in practice due to irreducible noise in real data.

---

## 4. Comparison of Models

| Model | Avg. Log-Likelihood | Interpretation |
|-------|---------------------|----------------|
| Random guess | $-\infty$ | No predictive capability |
| Null model ($\mu=0$, $\sigma=1$) | $-1.42$ | Baseline (no skill) |
| Our model ($\sigma \approx 0.07$) | $+1.15$ | Excellent predictions |
| Perfect model ($\sigma \to 0$) | $+\infty$ | Theoretical limit |

The progression can be visualized on a number line:

$$\underbrace{-\infty}_{\text{Random}} \quad \cdots \quad \underbrace{-1.42}_{\text{Null Model}} \quad \cdots \quad 0 \quad \cdots \quad \underbrace{+1.15}_{\text{Our Model}} \quad \cdots \quad \underbrace{+\infty}_{\text{Perfect}}$$

---

## 5. Interpretation on the Density Scale

The log predictive density of $1.15$ corresponds to a density value of:

$$p(y_i \mid \hat{y}_i, \sigma) \approx e^{1.15} \approx 3.16$$

> **Note:** For continuous distributions, probability *densities* can exceed 1 (unlike probabilities). A high density indicates that the predictive distribution is **narrow and well-centered** around the true observed values.

---

## 6. Relationship to Model Precision

The average log-likelihood directly relates to the model's predictive precision. For a Normal likelihood with unbiased predictions:

$$\mathbb{E}[\log p] = -\frac{1}{2}\log(2\pi) - \log(\sigma) - \frac{1}{2} = -\frac{1}{2}(1 + \log(2\pi\sigma^2))$$

This shows that smaller $\sigma$ (more precise predictions) leads to higher log-likelihood:

| $\sigma$ | $\mathbb{E}[\log p]$ |
|----------|----------------------|
| 1.00 | $-1.42$ |
| 0.50 | $-0.73$ |
| 0.20 | $+0.19$ |
| 0.10 | $+0.88$ |
| 0.07 | $+1.24$ |
| 0.05 | $+1.58$ |

---

## 7. Effective Number of Parameters

The $p_{\text{loo}} = 9.71$ represents the **effective number of parameters** in the model. This accounts for:

- The actual number of parameters
- The degree to which priors constrain the posterior
- The effective complexity used to fit the data

With 130 observations and approximately 10 effective parameters:

$$\text{Observations per parameter} = \frac{n}{p_{\text{loo}}} = \frac{130}{9.71} \approx 13$$

This ratio indicates sufficient data to support the model complexity.

---

## 8. Pareto k Diagnostics

The reliability of the LOO estimate is assessed using Pareto $k$ diagnostics:

| Range | Status | Count | Percentage |
|-------|--------|-------|------------|
| $k \leq 0.70$ | Good | 85 | 98.8% |
| $0.70 < k \leq 1.00$ | Bad | 1 | 1.2% |
| $k > 1.00$ | Very Bad | 0 | 0.0% |

With 98.8% of observations having $k \leq 0.70$, the overall LOO estimate is considered reliable. The single observation with $0.70 < k \leq 1.00$ indicates one potentially influential data point, which may warrant further investigation but does not invalidate the overall assessment.

---

## 9. Conclusion

Our model achieves an average log predictive density of $+1.15$, substantially better than the null model baseline of $-1.42$. This excellent performance is attributable to:

1. **Low residual variance**: $\sigma \approx 0.07$ (in standardized units)
2. **Unbiased predictions**: Residual means $\approx 0$ across all operating regimes
3. **Consistent performance**: Similar RMSE (0.064–0.075) across all setpoints

The LOO cross-validation confirms that the model generalizes well and is not overfitting to the training data.

---

## Quick Reference

> **elpd_loo Quick Reference**
>
> **Definition:** $\text{elpd\_loo} = \sum_{i=1}^{n} \log p(y_i \mid y_{-i})$
>
> **Interpretation:**
> - Higher values = better predictive performance
> - Positive values = better than null model
> - Per-observation: divide by $n$ for average
>
> **For Normal likelihood with residual std $\sigma$:**
> $$\mathbb{E}[\log p] \approx -0.919 - \log(\sigma) - 0.5$$
>
> **Benchmarks:**
> | $\sigma$ | $\mathbb{E}[\log p]$ |
> |----------|----------------------|
> | 1.0 (null) | $-1.42$ |
> | 0.1 | $+0.88$ |
> | 0.07 | $+1.24$ |

In [ ]:
# Compute PSIS weights (these reweight posterior samples to approximate LOO posterior)
weights, pareto_k = compute_psis_weights(-idata_top_data.log_likelihood_stacked)

In [ ]:
fig_elpd, trouble_obs_indices = evaluate_model_elpd(
    idata_top_data.y_obs,
    out_pressure_loo,
    idata_top_data.sigma_stacked.to_numpy(),
    idata_top_data.y_pred_stacked.to_numpy(),
    weights,
    pareto_k,
)

In [ ]:
plt.scatter(data_out_pressure.y_scaled.index, data_out_pressure.y_scaled)
plt.scatter(
    data_out_pressure.y_scaled.iloc[trouble_obs_indices].index,
    data_out_pressure.y_scaled.iloc[trouble_obs_indices],
    color="red",
    label="Poorly predicted",
)
plt.ylabel("Scaled outlet pressure")
plt.title("Scaled outlet pressure with poorly predicted points highlighted")
plt.legend()
plt.show()

# Bayesian LOO Diagnostic: Analytical Log-Density Calibration Check

## Overview

This diagnostic evaluates how well a Bayesian model's leave-one-out (LOO)
predictive performance aligns with what we would theoretically expect from a
perfectly calibrated Normal likelihood. By comparing the actual LOO
log-density against an analytically derived "ideal" distribution, we can
assess whether the model is well-calibrated, overconfident, or underconfident
in its predictions.

## Method

### Deriving the Effective Standardized Residual

For a Normal likelihood, the log-density of a single observation is:


$$
\log p(y_i \mid \mu, \sigma) = -\frac{1}{2}\log(2\pi\sigma^2)
- \frac{1}{2}\left(\frac{y_i - \mu}{\sigma}\right)^2
$$


Rearranging in terms of the average LOO log-density (ELPD / n), we can
back-solve for the root-mean-square (RMS) standardized residual:


$$
\left|\frac{y - \mu}{\sigma}\right|_{\text{RMS}}
= \sqrt{2\left(-\frac{1}{2}\log(2\pi\sigma^2)
- \overline{\text{loo}}_{\log}\right)}
$$


where $\overline{\text{loo}}_{\log}$ is the average pointwise LOO
log-density. For a perfectly calibrated Normal model, the expected value of
$(y - \mu)^2 / \sigma^2$ is 1.0, so the RMS z-score should be close to 1.

Using the posterior-average $\sigma$ and the computed ELPD, the resulting
RMS z-score was **1.04** — very close to the ideal value of 1.0, with the
small excess attributable to the expected penalty from out-of-sample
(leave-one-out) prediction.

### Analytical Log-Density Distribution

To visualize this comparison, we compute the "ideal" per-observation
log-density under the assumption of perfect calibration (RMS z-score = 1)
across all posterior draws of $\sigma$:


$$
\ell_{\text{ideal}}(\sigma) = -\frac{1}{2}\log(2\pi)
- \log(\sigma) - \frac{1}{2}
$$


This yields a distribution of ideal log-densities that reflects posterior
uncertainty in $\sigma$. We then overlay the actual average LOO log-density
to see where it falls relative to this distribution.

## Results

### Analytical Log-Density Calibration

The histogram shows the distribution of analytical (ideal)
log-densities across posterior samples of $\sigma$. The black vertical line
marks the posterior mean of this distribution, and the red vertical line
marks the actual average LOO log-density.

Key observations:

- **The LOO log-density (red) falls slightly to the left of the analytical
  mean (black)**, indicating that out-of-sample predictive performance is
  marginally worse than the in-sample ideal. This is expected behavior — LOO
  predictions leave each observation out, so a small degradation relative to
  the ideal is normal.

- **The LOO log-density lies well within the bulk of the analytical
  distribution**, meaning the observed out-of-sample performance is
  consistent with what the model's posterior uncertainty would predict under
  perfect calibration.

- **The RMS z-score of 1.04** corroborates this visual finding: the model's
  predictive distributions are neither too narrow (overconfident) nor too
  wide (underconfident).

### Pointwise ELPD Contributions

To examine the model's predictive performance at the individual observation
level, we exponentiate the pointwise LOO log-densities to obtain the
predictive density assigned to each observation:


$$
p(y_i \mid y_{-i}) = \exp(\text{elpd}_i)
$$


These are plotted against the observed values on a log-scale y-axis, along
with two reference lines:

- **Red dashed line (3.14)**: The geometric mean LOO density — the actual
  average out-of-sample predictive performance, computed as
  $\exp(\overline{\text{elpd}})$.
- **Black dashed line (3.26)**: The expected density under perfect
  calibration — what we would theoretically expect if every observation had
  an RMS z-score of exactly 1.0.


The small gap between the two reference lines (3.14 vs. 3.26, roughly a
3.7% decrease) represents the **out-of-sample prediction penalty** — the
natural cost of predicting each observation without having seen it during
fitting. This is entirely consistent with the RMS z-score of 1.04.

#### Interpreting the Scatter

Each point represents how well the model predicted that specific observation
when it was left out. The density value can be thought of as a measure of
"how surprised the model is" by that observation:

| Position | Meaning |
|:--------:|---------|
| Above the lines | The model found this observation **easy to predict** — it fell near the center of the LOO predictive distribution ($\|z\| < 1$) |
| Near the lines | The observation was about as surprising as a **typical draw** from a Normal distribution |
| Below the lines | The model found this observation **hard to predict** — it fell in the tails of the LOO predictive distribution ($\|z\| > 1$) |

This spread is **natural and expected**. Even for a perfectly calibrated
Normal model, the standardized residuals $(y_i - \mu)/\sigma$ follow a
standard Normal distribution, which means:

- ~38% of observations will have $|z| < 0.5$ → high density (above the
  lines)
- ~30% will have $0.5 < |z| < 1.0$ → near the lines
- ~24% will have $1.0 < |z| < 2.0$ → below the lines
- ~8% will have $|z| > 2.0$ → well below the lines

The reference lines represent average behavior — individual observations
will always scatter around them.

#### What to Watch For

Signs of potential problems in a pointwise ELPD plot include:

| Pattern | Concern |
|---------|---------|
| Points below the line concentrated at specific observed values | Model misspecification in those regions |
| More low-density outliers than the expected ~8% | Heavier tails than the Normal likelihood assumes |
| Too little spread (all points on the line) | Possible overfitting or an overly flexible model |
| Systematic trend with observed value | Model is miscalibrated in certain regions of the outcome space |

#### Assessment

The pointwise ELPD plot shows healthy behavior:

- ✅ The scatter is **roughly symmetric** around the reference lines.
- ✅ There is **no systematic pattern** with the observed value — the model
  is not miscalibrated in any particular region.
- ✅ Only **2–3 points** fall notably below the bulk, consistent with
  normal tail behavior (~8% expected).
- ✅ The majority of observations cluster at or above the geometric mean
  density.
- ⚠️ Three observations drop
  below $10^0$ in density. These may warrant further investigation and
  cross-referencing with Pareto $\hat{k}$ diagnostics to ensure the LOO
  approximation is reliable for those points.

## Summary

| Metric | Value | Interpretation |
|:------:|:-----:|----------------|
| RMS z-score | 1.04 | Very close to ideal (1.0); minor out-of-sample penalty |
| Geometric mean LOO density | 3.14 | Close to ideal (3.26); 3.7% out-of-sample penalty |
| LOO position in analytical distribution | Within bulk | LOO performance consistent with posterior uncertainty |
| Pointwise scatter pattern | Homogeneous | No systematic miscalibration across observed values |

## Interpretation Guide

| RMS Z-Score | LOO vs. Analytical | Pointwise Pattern | Interpretation |
|:-----------:|:------------------:|:------------------:|----------------|
| ≈ 1.0 | Red line near black line | Symmetric scatter, few outliers | Well-calibrated model |
| >> 1.0 | Red line far left of distribution | Many points below the line | Overconfident or misspecified model |
| << 1.0 | Red line far right of distribution | Most points above the line | Underconfident model (overly wide predictions) |

In [ ]:
fig_calibration, stats_calibration = plot_loo_calibration_curves(idata_top, RANDOM_SEED)


## Model calibration Assessment Summary 

All three diagnostics — the quantitative RMS z-score (1.04), the analytical
log-density histogram, and the pointwise ELPD contribution plot — converge
on the same finding: the Normal likelihood model is **well-calibrated**. The
leave-one-out predictive performance closely matches the analytically
expected log-density, with only a minor and expected out-of-sample penalty.
The pointwise analysis confirms that no individual observations or regions
of the outcome space are systematically problematic. This suggests the model
is neither overfitting nor producing overly conservative uncertainty
estimates. Also the LOO Probability Integral Transform (PIT) values are quite uniformly distributed.
Finally the intutive calibration plot confirms that the emprical coverage intervals 
calculated from model PIT values follow diagonal trend against the corresponding 
expected coverage intervals considering the finite-sampling and Bayesian
bootstrap uncertainty bands with. 

# Bayesian Model Evaluation: Understanding Uncertainty and Metrics

Assume a standard regression model:

$$y \sim \mathcal{N}(\mu, \sigma) \\
\mu = \mathcal{f}\left(X, \beta \right)=\beta_{0} + \beta_{1}.X_{1}+\beta_2.X_{2}+...$$

Two Types of Uncertainty

1. Posterior Uncertainty (Epistemic Uncertainty): Uncertainty about the model parameters ($\theta$)
2. Observation-Level Prediction Uncertainty (Aleatoric Uncertainty): Variability in predictions across different observations


$\text{trace}.\text{posterior}[\mu]$ is the deterministic noise function (expected value): $\mu=E[y | X, \theta]$

 - No random noise included
 - Pure systematic component
 - Each draw $\mu_{d} = \mathcal{f}\left(X, \beta_{d} \right)$ where $d$ indexes the posterior draw.

$\text{trace}.\text{posterior\_predictive}[y_{obs}]$ is the full predictive distribution including observation noise:
$$y_{pred} \sim \mathcal{N}(\mu, \sigma) \\
y_{pred} = \mu + \sigma.\epsilon \qquad \text{where} \quad\epsilon \sim \mathcal{N}(0, 1)$$
- Stochastic predictions (includes observation noise). Note the random seed provided for sampling, otherwise for a partcular draw, we can get different noise values.
- Full data-generating process
- Each draw represmets a complete observation (signal + noise): $y_{d}= \mathcal{f}\left(X, \beta_{d} \right) + \sigma_{d}.\epsilon_{d}$ where $d$ indexes the posterior draw.

Variance and Expected Value decomposition

$Var\left(y_{pred}\right)=Var\left(\mu\right) + Var\left(\sigma.\epsilon\right)[\sigma^2.Var\left(\epsilon\right)=\sigma^2.1]=Var\left(\mu\right)+\sigma^2$
 - Var $\left(\mu \right)$: Epistemic uncertainty (parameter uncertainty) or variability due to systematic component (explained variance)
 - $\sigma^2$: Aleatoric uncertainty (irreducible noise) or variability due to random noise (unexplained variance)
 - Var $\left(y_{pred} \right)$: Total uncertanity

$E[y_{pred}] \left(E[\mu+\sigma.\epsilon]=E[\mu]+\sigma.E[\epsilon]=E[\mu]+\sigma.0 \right) = E[\mu] = \mu$

### Compare emprical σ to the model’s implied observation noise σ ($y_{obs} - \mu$)

Take important note that we should not use posterior predictive samples to calculate emprical residuals here as:

\begin{align*}
y_{\mathrm{pred}} - y_{\mathrm{obs}} &= \left(\mu_{\mathrm{posterior}} + \sigma_{\mathrm{posterior}} \cdot \varepsilon_{\mathrm{pred}}\right) - \left(\mu_{\mathrm{true}} + \sigma_{\mathrm{true}} \cdot \varepsilon_{\mathrm{obs}}\right) \\
&= \left(\mu_{\mathrm{posterior}} - \mu_{\mathrm{true}}\right) + \left(\sigma_{\mathrm{posterior}} \cdot \varepsilon_{\mathrm{pred}} - \sigma_{\mathrm{true}} \cdot \varepsilon_{\mathrm{obs}}\right) \\
&= \text{systematic error} + \text{noise}_{\mathrm{pred}} - \text{noise}_{\mathrm{obs}}
\end{align*}

\begin{align*}
\text{Var}(y_{\mathrm{pred}} - y_{\mathrm{obs}}) &= \text{Var}(\mu_{\mathrm{posterior}} - \mu_{\mathrm{true}}) + \text{Var}(\sigma_{\mathrm{posterior}} \cdot \varepsilon_{\mathrm{pred}}) + \text{Var}(\sigma_{\mathrm{true}} \cdot \varepsilon_{\mathrm{obs}}) \\
&\approx \text{Var}(\mu_{\mathrm{posterior}} - \mu_{\mathrm{true}}) + \sigma_{\mathrm{posterior}}^2 + \sigma_{\mathrm{true}}^2
\end{align*}

If $\sigma_{posteriot} \approx \sigma_{true} \approx \sigma$:
$$\text{Var}\left(y_{pred} - y_{obs}\right) \approx \text{Var}\left(\text{systematic error}\right) + 2 \sigma^2$$

To check our noise model, we need to use only posterior uncertanity as:
\begin{align*}
\left(\mu_{posterior} - y_{\mathrm{obs}}\right) &= \mu_{\mathrm{posterior}} -\left(\mu_{true} + \sigma_{\mathrm{true}} \cdot \epsilon_{obs}\right)\\
&= \left(\mu_{\mathrm{posterior}} - \mu_{\mathrm{true}}\right) - \sigma_{\mathrm{true}} \cdot \varepsilon_{\mathrm{obs}} \\
&= \text{systematic error} - \text{noise}_{\mathrm{obs}}
\end{align*}
Now if we get the variance:
\begin{align*}
\operatorname{Var}\left(\mu_{posterior} - y_{\mathrm{obs}}\right) &= \operatorname{Var}(\mu_{\text{posterior}} - \mu_{\text{true}}) + \operatorname{Var}(\sigma_{\text{true}} \cdot \varepsilon_{\text{obs}}) \\
&\approx \operatorname{Var}(\text{systematic error}) + \sigma_{\text{true}}^2
\end{align*}



Take important note that the raw $\sigma$ (dotted line) includes BOTH between-setpoint variance AND within-setpoint noise while the model $\sigma$ estimates only the within-setpoint noise after accounting for the mean function $\hat{\mu}$. For ideal calibration: The model $\sigma$ should capture the residual noise after accounting for $\mu$, NOT match the raw $\sigma$!

$$
\begin{align}
\sigma_{\text{raw}} &= \sqrt{\sigma^2_{\mu} + \sigma^2_{\epsilon}} \\
\sigma_{\text{model}} &= \sigma_{\epsilon}
\end{align}
$$

$$
\begin{aligned}
\text{Where:} \quad & \\
\sigma^2_{\mu} &= \text{variance explained by the mean function (between-setpoint)} \\
\sigma^2_{\epsilon} &= \text{residual variance (within-setpoint measurement noise)}
\end{aligned}
$$

$$
\begin{aligned}
\text{Model estimate:} \quad & \sigma_{\text{model}} = E[\sigma \mid \text{data}] \\
\text{Empirical check:} \quad & \sigma_{\text{residual}} = \text{SD}(y - \hat{\mu}) \\
\text{Ideal calibration:} \quad & \sigma_{\text{model}} \approx \sigma_{\text{residual}}
\end{aligned}
$$


- **Model Assessment**
  - **Mean Function (μ): EXCELLENT**
    - Captures 99.5% of total variance
    - Correctly models between-setpoint effects
  
  - **Noise Model (σ): WELL-CALIBRATED**
    - Global σ is appropriate (residual SD is ~flat across setpoints)
    - Ratio ≈ 0.94 indicates conservatism (acceptable)
    - All empirical residuals fall within posterior credible intervals
  
  - **Recommendation: NO CHANGES NEEDED**
    - Keep global σ

- **Model is ready for inference/prediction**

Why Bayesian R² Uses Posterior Uncertainty?

This is asking: "What is the distribution of R² values given my uncertainty about the parameters?" It's fundamentally about model uncertainty, not about individual predictions. For a given posterior draw of the model indexed $d$, what fraction of the variability across observations is explained by the model’s mean function?
$$R^2 = \frac{\mathrm{Var}\left(\mu_d \right)}{\mathrm{Var}\left(\mu_d \right) + \mathrm{Var}\left(y_{obs}-\mu_d \right)}$$

In [ ]:
model_non_hierarchical_sigma_eval = evaluate_noise_model(
    data_out_pressure,
    idata_top_data,
    std_unscaled,
    RANDOM_SEED,
)

### Demonstrate hierarchical sigma model

In [ ]:
idata_hierarchical_noise_data = exctract_pymc_groups_data(idatas_dict["hierarchical_noise"])

model_hierarchical_sigma_eval = evaluate_noise_model(
    data_out_pressure,
    idata_hierarchical_noise_data,
    std_unscaled,
    RANDOM_SEED,
)

# Posterior uncertanity versus pointwise predictive error for each observation

Posterior uncertanity captures: "How uncertain am I about the model's predictive performance?"

 - Reflects uncertainty in the parameters (β coefficients, σ, etc.)
 - Each draw represents a plausible model configuration
 - The distribution shows how MAE varies across different plausible models
 - Analogous to frequentist bootstrap uncertainty in performance metrics

While observation-level prediction uncertanity captures "Which individual predictions are more/less certain?"
- Shows which specific observations are hard to predict
- Uses the posterior mean (or median) as the point prediction
- The distribution is across observations, not posterior draws
- Useful for diagnostics and identifying outliers

The Analogy

Posterior uncertainty = "If I fit this model to different bootstrap samples, how much would my overall prediction errors vary?"

Observation uncertainty = "For this particular fitted model, which individual data points have larger prediction errors?"

In [ ]:
_ = evaluate_model_performance(
    idata_top,
    idata_top_data,
    setpoint_timeseries=data_out_pressure.setpoint_timeseries,
    random_seed=RANDOM_SEED,
)

The residuals vs fitted values plot demonstrates that our model captures the underlying relationships well. Residuals are randomly distributed around zero with no systematic patterns, roughly constant variance across the range of fitted values, and similar behavior across all setpoints (SP = 80, 100, 140, 180). Also, all residuals fall within the HDI bounds, indicating good model fit.

# Implement cusum

In [ ]:
X_full_scaled: pd.DataFrame = (X - data_out_pressure.X_mean) / data_out_pressure.X_std
y_full_scaled: pd.Series = (y - data_out_pressure.y_mean) / data_out_pressure.y_std

In [ ]:
with model_top:
    pm.set_data(
        {"X": X_full_scaled.to_numpy()},
        coords={"obs": np.arange(len(X_full_scaled))},
    )
    post_pred_full = pm.sample_posterior_predictive(
        idata_top, predictions=True, random_seed=RANDOM_SEED
    )

In [ ]:
y_pred_full_raw = post_pred_full.predictions["y_like"]
y_pred_full_raw.loc[{'obs': shutin_mask.to_numpy()}] = np.nan

plot_predictions_with_uncertainty(
    y_pred_full_raw,
    y_full_scaled.mask(shutin_mask).to_numpy(),
    y_pred_full_raw.stack(sample=("chain", "draw")).mean(axis=1)  # noqa: PD013 - stack() required for xarray dimension stacking
)

The 95% HDI coverage of 58% (with average width 0.30) initially suggests overconfidence, but this is expected because the model was trained on clean baseline data (130 points) and applied to a full timeseries (206 points) that includes ~76 anomalous points. Anomalous points come from a different distribution and should fall outside the intervals — that's the goal. The math checks out: if ~95% of clean points land inside (~124/130) and ~0% of anomalies do, total coverage is ~60%, matching 58%. The model is likely well-calibrated on in-distribution data and successfully flagging outliers; to confirm, one can compute coverage separately for clean and anomalous regimes.

In [ ]:
residual_full_distribution = (
    y_full_scaled.to_numpy()
    - y_pred_full_raw.stack(sample=("chain", "draw")).T.to_numpy()  # noqa: PD013 - stack() required for xarray dimension stacking
)  # (chain * draw, n_obs)
full_data_timestamps: pd.DatetimeIndex = cast(pd.DatetimeIndex, y_full_scaled.index)

In [ ]:
# Calculate CUSUM for negative drift
uncertain_cusum_result = calculate_cusum_with_uncertainty(
    residuals_distribution=residual_full_distribution,
    timestamps=full_data_timestamps,
    shutin_mask=shutin_mask,
    baseline_mask=baseline_mask,
    anomaly_direction="neg",
    drift_multiplier=0.5,
    # n_samples_subset=1000  # Use 1000 samples for speed, or None for all
)

In [ ]:
cusum_paths = uncertain_cusum_result.cusum_paths
cusum_mean = uncertain_cusum_result.mean
healthy_indices = np.where(baseline_mask)[0]
cusum_healthy = cusum_paths[:, healthy_indices]
# Maximum absolute CUSUM during healthy period (per sample)
max_cusum_healthy = np.max(np.abs(cusum_healthy), axis=1)

# Control limits as percentiles of max excursion
h_90 = np.percentile(max_cusum_healthy, 90)
h_95 = np.percentile(max_cusum_healthy, 95)
h_99 = np.percentile(max_cusum_healthy, 99)

print("\nControl Limits (from healthy period):")
print(f"  h_90: ±{h_90:.2f}")
print(f"  h_95: ±{h_95:.2f}")
print(f"  h_99: ±{h_99:.2f}")

In [ ]:
# Compute masked statistics
residual_stats = compute_masked_statistics(
    residual_full_distribution, full_data_timestamps, shutin_mask
)

# Compute exceedance probabilities
h = h_95  # Use 95th percentile control limit
p_above, p_below, p_either = compute_exceedance_probability(cusum_paths, h)


# Find first detection time (when probability exceeds threshold)
detection_threshold = 0.95  # Signal when 95% of samples exceed limit

# For positive drift (CUSUM going up)
detect_mask_lower = p_above > detection_threshold

if detect_mask_lower.any():
    first_detection = full_data_timestamps[detect_mask_lower][0]
    detection_idx = np.where(detect_mask_lower)[0][0]
    print("\n🚨 First anomaly detection (negative drift):")
    print(f"   Time: {first_detection}")
    print(f"   CUSUM mean at detection: {cusum_mean[detection_idx]:.2f}")
    print(f"   P(CUSUM > h): {p_above[detection_idx]:.1%}")

In [ ]:
fname = Path(IMAGE_DIR / "fouling_summary_probabilistic.png")
visualize_probabilistic_cusum(
    residual_stats, uncertain_cusum_result, full_data_timestamps, h_95, p_above, save=True, fname=fname
)

CUSUM Uncertainty Width Grows during anomaly periods (more uncertainty as CUSUM accumulates) and Resets after shutdown (as expected)

## Uncertainty-Aware CUSUM Results

The Bayesian framework successfully detected two distinct anomaly events:

### Anomaly #1 (November 2024)
- **Detection**: Mid-November 2024
- **Severity**: Moderate (CUSUM reached ~7, approximately 12× threshold)
- **Cause**: Outlet pressure consistently below expected values
- **Resolution**: Successfully resolved by December maintenance

### Anomaly #2 (March-April 2025)  
- **Detection**: Late February 2025
- **Severity**: Critical (CUSUM reached ~20 and rising, approximately 33× threshold)
- **Cause**: Outlet pressure significantly below expected values
- **Status**: Ongoing - requires immediate attention

### Key Advantages of Uncertainty-Aware CUSUM
1. **Memoryless after maintenance**: CUSUM correctly resets, avoiding false alarms
2. **Full uncertainty quantification**: 95% credible intervals on CUSUM paths
3. **Probabilistic detection**: Clear transition from 0% to 100% anomaly probability
4. **Validated baseline**: Per-sample parameters from calibrated Bayesian model